# Lezione 01 — Dal notebook alla produzione

**Corso:** MLOps e Monitoraggio con Python · **Blocco 1:** Il problema e le basi
**Tempo stimato:** 90 minuti · **Capitolo del manuale:** `manual/01_del_notebook_a_produccion_IT.md`

## Cosa vedrai in questa lezione

- Il **ciclo di vita** di un sistema di ML e perché "il modello" è solo un pezzo piccolo.
- Il **debito tecnico specifico del ML** (Sculley et al., 2015): accoppiamento delle configurazioni, dipendenze dai dati nascoste, *entanglement*, *training-serving skew*.
- I **livelli di maturità MLOps** di Google Cloud e di Microsoft Azure, confrontati.
- Un'**ispezione guidata, con codice**, del notebook `datos/notebook_sucio.ipynb`: lo apriamo come dati (senza eseguirlo) e contiamo i suoi problemi.
- Una **dimostrazione dal vivo**: lo stesso calcolo dà un risultato diverso a seconda di quale cella è stata eseguita per ultima.
- L'**accuratezza ingannevole**: con classi sbilanciate, un modello che non impara nulla può sembrare che "funzioni bene".

Questo notebook **non addestra nessun modello**: è deliberatamente leggero. Il notebook sporco che analizziamo è materiale di diagnostica, non qualcosa che dobbiamo eseguire dall'inizio alla fine.

## Prerequisiti

- Nessuna libreria nuova: `numpy` e la libreria standard (`json`, `os`, `re`). La cella seguente installa solo ciò che manca (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso, con `notebook_sucio.ipynb` all'interno. In Colab, caricala dal pannello dei file.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta.

---
## Installazione

Installa solo ciò che manca. In un ambiente con tutto installato non fa nulla.

In [ ]:
import subprocess, sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
print("Librerie pronte.")

---
## Cella 0 — Setup

Definiamo `cargar_datos` e apriamo `notebook_sucio.ipynb` **come dati**: lo leggiamo con `json.load`, esattamente come faresti con qualsiasi file JSON. Non lo eseguiamo. Più avanti, nella dimostrazione dal vivo, useremo un generatore di numeri casuali con un proprio seme (`np.random.default_rng(42)`), quindi qui non serve fissare nessun seme globale.

In [ ]:
import os
import json
import re

import numpy as np

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

RUTA_SUCIO = cargar_datos("notebook_sucio.ipynb")
sucio = json.load(open(RUTA_SUCIO, encoding="utf-8"))
print("Notebook caricato:", RUTA_SUCIO)
print("Celle totali:", len(sucio["cells"]))

---
## 1. Il ciclo di vita e "il modello non è il sistema"

Prima di guardare il notebook sporco, ricorda la mappa del capitolo: **idea → dati → addestramento → validazione → distribuzione → monitoraggio → manutenzione**, in un anello. Il notebook che ispezioneremo cerca di coprire solo "dati" e "addestramento", una volta sola, senza anello e senza le altre cinque fasi. Questa è, di per sé, la prima osservazione dell'analisi.

---
## 2. Ispezione della struttura, senza eseguire nulla

Contiamo celle di codice e di markdown, e cerchiamo con un'espressione regolare gli indizi più facili da rilevare meccanicamente: percorsi assoluti di un sistema operativo specifico.

In [ ]:
tipos = [celda["cell_type"] for celda in sucio["cells"]]
n_code = tipos.count("code")
n_md = tipos.count("markdown")
print("codice:", n_code, "| markdown:", n_md)

PATRON_RUTA = re.compile(r"(/Users/\w+|/home/\w+)[\w/.]*")
rutas = []
for i, celda in enumerate(sucio["cells"]):
    if celda["cell_type"] != "code":
        continue
    fuente = "".join(celda["source"])
    for m in PATRON_RUTA.finditer(fuente):
        rutas.append((i + 1, m.group(0)))

print("percorsi assoluti trovati:", len(rutas))
for celda_n, ruta in rutas:
    print("  cella", celda_n, "→", ruta)

**Cosa osservi?** Il notebook ha 16 celle (11 di codice, 5 di markdown) e contiene **2 percorsi assoluti hardcoded**, entrambi del portatile di una persona specifica: uno per caricare un file "extra" (cella 3) e l'altro per salvare il modello (cella 16). Nessuno dei due funzionerebbe sulla tua macchina così com'è.

---
## 3. Dimostrazione dal vivo: l'ordine di esecuzione cambia il risultato

Nel notebook sporco, la cella 6 fissa `umbral_riesgo = 0.5`, la cella 8 lo sovrascrive a `0.8` "per provare", e la cella 9 usa `umbral_riesgo` per contare i clienti a rischio. Il risultato della cella 9 dipende da **quale delle due celle (6 o 8) è stata eseguita per ultima**, non dall'ordine in cui appaiono a schermo.

Riproduciamo qui lo stesso meccanismo con dati minimi e un seme fisso, per vederlo con numeri concreti e riproducibili (a differenza del notebook sporco, qui controlliamo davvero la casualità).

In [ ]:
rng = np.random.default_rng(42)
prob_estimada_demo = rng.random(20)          # 20 "clienti" giocattolo, probabilità di abbandono simulata


def contar_en_riesgo(umbral):
    """Stesso codice della cella 9 del notebook sporco, isolato in una funzione."""
    return int((prob_estimada_demo > umbral).sum())


# Ordine A: l'ultima cella "di soglia" eseguita ha fissato 0.5
umbral_riesgo = 0.5
resultado_orden_a = contar_en_riesgo(umbral_riesgo)

# Ordine B: qualcuno riesegue la cella di prova con 0.8, senza toccare il resto
umbral_riesgo = 0.8
resultado_orden_b = contar_en_riesgo(umbral_riesgo)

print("stesso notebook, stesso codice di conteggio:")
print("  umbral_riesgo = 0.5  →", resultado_orden_a, "clienti a rischio")
print("  umbral_riesgo = 0.8  →", resultado_orden_b, "clienti a rischio")
assert resultado_orden_a != resultado_orden_b

**Cosa osservi?** Con gli stessi 20 clienti giocattolo, `contar_en_riesgo` restituisce 12 clienti a rischio se l'ultima cella di soglia eseguita è stata quella di `0.5`, e 5 se è stata quella di `0.8`. Il codice della cella 9 (qui, `contar_en_riesgo`) non è cambiato in nessun momento: l'unica cosa che è cambiata è **quale cella è stata eseguita per ultima**. Questo è esattamente il sintomo di *entanglement* della sezione 4 del capitolo: cambiare "solo una cella" altera un risultato che appare diverse celle dopo, senza nessun avviso.

---
## 4. L'accuratezza ingannevole: un altro problema che non si vede a prima vista

La cella 14 del notebook sporco stampa `modelo.score(...)` (accuratezza) come se quella cifra bastasse a dire "il modello funziona". Non è così: se l'abbandono è minoritario, un modello che quasi non predice mai "abbandona" può avere un'accuratezza altissima senza aver imparato nulla di utile. Lo verifichiamo con un esempio minimo, senza addestrare nulla.

In [ ]:
# 100 clienti giocattolo, 12 dei quali abbandonano (esempio illustrativo, senza addestrare nulla)
abandono_ejemplo = np.array([0] * 88 + [1] * 12)
prediccion_trivial = np.zeros_like(abandono_ejemplo)          # "non predice mai abbandono"

exactitud_trivial = (prediccion_trivial == abandono_ejemplo).mean()
abandonos_reales = int(abandono_ejemplo.sum())
abandonos_detectados = int(((prediccion_trivial == 1) & (abandono_ejemplo == 1)).sum())

print("tasso di abbandono:", abandono_ejemplo.mean())
print("accuratezza di un modello che non predice mai abbandono:", exactitud_trivial)
print(f"dei {abandonos_reales} abbandoni reali, quel modello ne rileva:", abandonos_detectados)
assert exactitud_trivial > 0.85 and abandonos_detectados == 0

**Cosa osservi?** Un modello che **non predice mai abbandono** indovina l'88 % delle volte (perché l'88 % dei clienti, infatti, non abbandona) e rileva 0 dei 12 abbandoni reali. Se la cella 14 del notebook sporco guarda solo `modelo.score(...)`, un risultato così inutile può passare per "il modello funziona bene". Bisogna, come minimo, guardare quanti abbandoni reali rileva il modello (si riprende con metriche proprie nella lezione 6).

---
## 5. A che livello di maturità si trova questo progetto?

Con quanto abbiamo appena osservato, e con il resto dei problemi descritti nel capitolo (nessun seme nella generazione dei dati né nel `train_test_split`, una cella che funziona solo in un ordine diverso da quello visuale, una dipendenza dai dati nascosta, un modello addestrato e buttato via, zero prove automatiche), il notebook sporco è un esempio di **Livello 0** sia nella scala di Google Cloud (tutto manuale, senza *pipeline*) sia in quella di Microsoft Azure (Livello 0 — Nessun MLOps). La lezione 2 riprende questo stesso file e comincia a farlo salire di livello.

---
## Riepilogo della lezione

- Un **sistema di ML** è molto più di "il modello": dati, configurazione, servizio, monitoraggio e manutenzione, in un ciclo che si ripete.
- Il **debito tecnico del ML** ha forme proprie: accoppiamento delle configurazioni, dipendenze dai dati nascoste, *entanglement*, *training-serving skew*.
- I **livelli di maturità MLOps** (Google Cloud: 3 livelli; Azure: 5 livelli) collocano un progetto in base a quanto del suo ciclo di vita è automatizzato.
- Il notebook `datos/notebook_sucio.ipynb` è Livello 0 da manuale: percorso assoluto hardcoded, senza semi, una cella che dipende da un ordine di esecuzione diverso da quello visuale, una variabile globale sovrascritta, una dipendenza dai dati nascosta, senza validazione dei dati, un'accuratezza che non dice nulla su se il modello abbia imparato qualcosa, un modello che si addestra e si butta, e zero prove.

*Continua con la lezione 2: Riproducibilità.*